# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
import re

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
reg_model = ''
runs = '-1-'
comment = "acute_all-compete_d_E"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
#sec_diff_bias_list = []
cell_series_list = []
# lineage_diff_nets = []

In [3]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/'
run_list = [int(re.search('sim_batch_(.*?)\.', f).group(1)) for f in os.listdir(d_rerun) if 'sim_batch' in f and comment in f and runs in f]
out = [str(x) for x in [k for k in np.arange(0, 1248)] if x not in run_list]
print(len(out))
print(' '.join((out)))

0



In [4]:
num_cpu = 60
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and comment in f and 'sim_batch' in f]
num_files = len(file_list)

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    parameters = np.array(import_dict["parameters"])
    sim_sum = np.array(import_dict["summary_stats"])

    out = np.hstack((parameters, sim_sum))

    return out

# create dataframe of infection response statistics
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
# mean_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
#                                                                                                         for file_name in file_list)), 
#                        columns = [i for i in var_names]).groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean()
full_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
                                                                                                        for file_name in file_list)), 
                       columns = [i for i in var_names])

# # Save datasets
full_df.to_pickle(os.path.join(d, "raw", "stacked_full_data"+runs+"runs"+'-'+comment)+'.pkl')

In [5]:
with pd.option_context('display.max_columns', None):
    display(full_df)

,S_0,I_0,b_I,d_S,d_I,d_IE,K_I,d_H,K_H,K_S,N_0,max_Na,t_bind,t_unbind,t_Na_div,t_E_div,t_M_div,t_E_die,t_cycle,psi_myc_I,psi_myc_HI,psi_myc_HE,L0_Na,psi_NE_I,psi_NE_HI,psi_NE_HE,L0_NE,psi_EM_I,psi_EM_HI,psi_EM_HE,L0_EM,psi_Edie_I,psi_Edie_HI,psi_Edie_HE,L0_Edie,p_load,T_max_pI,T_min_pI,harm_pI,harm_pS,max_pE,T_pE_max,T_pE_start,max_eM,T_pEcyteM,T_pE_end,frac_cM,int_pHE,int_pHI,E_end
0,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,10000000.0,100.0,4.0,1.0,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.333333,0.333333,-1.666667,1.0,-0.333333,0.333333,-1.666667,1.0,-0.0,0.0,-0.0,-81.0,0.333333,-0.333333,1.666667,-3.0,3.979039e-16,0.00,0.02,1.010000e+03,5.871328e+06,69964.0,4.46,1.33,0.0,0.0,30.00,0.190625,301135.577448,2.529497e+02,49367.0
1,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,10000000.0,100.0,4.0,1.0,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.333333,0.333333,-1.666667,1.0,-0.333333,0.333333,-1.666667,1.0,-0.0,0.0,-0.0,-81.0,0.333333,-0.333333,1.666667,-2.5,3.979039e-16,0.00,0.02,1.010000e+03,4.748767e+06,55922.0,4.44,1.27,0.0,0.0,30.00,0.178788,242706.242766,2.524598e+02,42444.0
2,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,10000000.0,100.0,4.0,1.0,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.333333,0.333333,-1.666667,1.0,-0.333333,0.333333,-1.666667,1.0,-0.0,0.0,-0.0,-81.0,0.333333,-0.333333,1.666667,-2.0,4.547474e-16,0.00,0.02,1.010000e+03,3.838786e+06,42726.0,4.36,1.04,0.0,0.0,30.00,0.182692,187735.503883,2.515841e+02,32173.0
3,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,10000000.0,100.0,4.0,1.0,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.333333,0.333333,-1.666667,1.0,-0.333333,0.333333,-1.666667,1.0,-0.0,0.0,-0.0,-81.0,0.333333,-0.333333,1.666667,-1.5,2.842171e-16,0.00,0.02,1.010000e+03,2.899993e+06,30191.0,4.89,1.46,0.0,0.0,30.00,0.213296,139252.957140,2.528578e+02,18777.0
4,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,10000000.0,100.0,4.0,1.0,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.333333,0.333333,-1.666667,1.0,-0.333333,0.333333,-1.666667,1.0,-0.0,0.0,-0.0,-81.0,0.333333,-0.333333,1.666667,-1.0,3.410605e-16,0.00,0.02,1.010000e+03,2.114811e+06,20372.0,5.09,1.31,0.0,0.0,30.00,0.205047,98249.004020,2.522688e+02,14762.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390971524,10000000.0,1000.0,2.500000e-07,0.01,0.5,12.0,10000000.0,1.0,100000.0,10000000.0,100.0,4.0,1.0,1.0,0.366667,0.333333,0.5,10.0,0.25,0.333333,-1.000000,0.333333,1.0,0.333333,-1.000000,0.333333,3.0,0.0,-0.0,0.0,-81.0,-0.333333,1.000000,-0.333333,1.5,4.600001e-02,5.42,23.86,9.931190e+06,1.312425e+03,105.0,2.59,1.45,0.0,0.0,2.64,0.226601,422.612433,1.987413e+06,0.0
390971525,10000000.0,1000.0,2.500000e-07,0.01,0.5,12.0,10000000.0,1.0,100000.0,10000000.0,100.0,4.0,1.0,1.0,0.366667,0.333333,0.5,10.0,0.25,0.333333,-1.000000,0.333333,1.0,0.333333,-1.000000,0.333333,3.0,0.0,-0.0,0.0,-81.0,-0.333333,1.000000,-0.333333,2.0,4.599813e-02,5.42,23.86,9.931390e+06,1.118522e+03,113.0,2.54,1.80,0.0,0.0,2.59,0.208437,416.710693,1.987477e+06,0.0
390971526,10000000.0,1000.0,2.500000e-07,0.01,0.5,12.0,10000000.0,1.0,100000.0,10000000.0,100.0,4.0,1.0,1.0,0.366667,0.333333,0.5,10.0,0.25,0.333333,-1.000000,0.333333,1.0,0.333333,-1.000000,0.333333,3.0,0.0,-0.0,0.0,-81.0,-0.333333,1.000000,-0.333333,2.5,4.600003e-02,5.42,23.86,9.931143e+06,1.358210e+03,123.0,2.14,1.82,0.0,0.0,2.55,0.198511,441.988318,1.987396e+06,0.0
390971527,10000000.0,1000.0,2.500000e-07,0.01,0.5,12.0,10000000.0,1.0,100000.0,10000000.0,100.0,4.0,1.0,1.0,0.366667,0.333333,0.5,10.0,0.25,0.333333,-1.000000,0.333333,1.0,0.333333,-1.000000,0.333333,3.0,0.0,-0.0,0.0,-81.0,-0.333333,1.000000,-0.333333,3.0,4.599833e-02,5.42,23.86,9.931410e+06,1.099793e+03,101.0,2.70,1.72,0.0,0.0,2.70,0.153465,401.388819,1.987484e+06,0.0


In [6]:
# Create additional variables
virs = np.unique(full_df[['I_0','d_I','K_I','b_I','K_H','N_0']].values, axis = 0)

full_df['antigenicity_over_harm'] = antigenicity_over_harm(full_df)
full_df['stim_pI'] = np.log(1 + (full_df['p_load']/full_df['K_I']))
full_df['stim_pHI'] = np.log(1 + (full_df['int_pHI']/full_df['K_H']))
full_df['stim_pHE'] = np.log(1 + (full_df['int_pHE']/full_df['K_H']))
#full_df['scaled_min_pS'] = full_df['min_pS']/full_df['S_0']

# identify Biologically evidenced networks
keep_vars = ['harm_pI', 'harm_pS', 'max_pE',
             'T_pE_start', 'T_pE_max', 'T_pE_end',
             'stim_pI', 'stim_pHI', 'stim_pHE',
             'E_end', 'antigenicity_over_harm']

In [7]:
# save data sets
full_infection_scenarios = []
mean_of_infection_scenarios = []
std_of_infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_0

for l, (I_0, d_I, K_I, b_I, K_H, N_0) in enumerate(tqdm(virs)):
    data = full_df.loc[(full_df["d_I"] == d_I)*(full_df["K_I"] == K_I)*(full_df["b_I"] == b_I)*(full_df["K_H"] == K_H)*(full_df["N_0"] == N_0)*(full_df["I_0"] == I_0), 
    ['b_I','d_I', 'K_I', 'I_0','S_0', 'N_0', 'd_S', 'K_H'] + Na_reg + NE_reg + EM_reg + EE_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_I = K_I, d_I = d_I, b_I = b_I,
                                   infection_model = "cancer" if b_I >= b_C else "acute")
    no_eff_stats = no_eff_data[l]["summary_stats"]

    data.loc[:,"harm_pI_noprotection"] = no_eff_stats[3]/S_0
    data.loc[:,"peff_infection"] = data['harm_pI'].to_numpy()/S_0
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/S_0
    data.loc[:,"peff_total_harm"] = data['peff_infection'] + data['peff_toxicity']
    data.loc[:,"peff_scaled_total_harm"] = data['peff_total_harm']/data['harm_pI_noprotection']

    mean_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean())
    std_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).std())
    full_infection_scenarios.append(data)

# stack datasets
pd.concat(mean_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(std_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(full_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/processed_full_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(mean_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(std_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/list_processed_full_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(full_infection_scenarios, f)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 81/81 [58:49<00:00, 43.58s/it]


In [8]:
# Clear memory
del full_df, mean_of_infection_scenarios, std_of_infection_scenarios, full_infection_scenarios